In [2]:
import pandas as pd
from pathlib import Path

In [3]:
project_path = Path("..")
raw_path = project_path / "data" / "raw"
processed_path = project_path / "data" / "processed"

In [4]:
processed_path.mkdir(parents=True, exist_ok= True)

In [5]:
customers = pd.read_csv(raw_path / "customers.csv")
orders = pd.read_csv(raw_path / "orders.csv")

In [6]:
customers_clean = customers.copy()
orders_clean = orders.copy()

ORDERS TABLE

In [16]:
orders_clean.dtypes

order_id                                   str
customer_id                                str
order_date                      datetime64[us]
year                                     int64
month                                    int64
quarter                                    str
day_of_week                                str
product_name                               str
category                                   str
unit_price_usd                         float64
quantity                                 int64
subtotal_usd                           float64
discount_pct                             int64
discount_amount_usd                    float64
shipping_fee_usd                       float64
tax_pct                                  int64
tax_amount_usd                         float64
total_amount_usd                       float64
payment_method                             str
device_used                                str
delivery_days                            int64
delivery_date

In [12]:
orders_clean["order_date"] = pd.to_datetime(orders_clean["order_date"])

In [14]:
orders_clean["delivery_date"] = pd.to_datetime(orders_clean["delivery_date"])

In [17]:
orders_clean["returned"].value_counts(dropna=False)

returned
0    22980
1     2020
Name: count, dtype: int64

In [18]:
orders_clean["is_repeat_customer"].value_counts(dropna=False)

is_repeat_customer
1    16149
0     8851
Name: count, dtype: int64

In [19]:
numeric_cols = [
    "unit_price_usd",
    "quantity",
    "subtotal_usd",
    "discount_pct",
    "discount_amount_usd",
    "shipping_fee_usd",
    "tax_pct",
    "tax_amount_usd",
    "total_amount_usd",
    "delivery_days",
    "customer_rating",
    "session_duration_minutes",
    "pages_viewed_before_purchase"
]

orders_clean[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
unit_price_usd,25000.0,68.124030,57.258933,3.36,29.2500,51.530,87.8775,697.03
quantity,25000.0,1.695520,1.045436,1.00,1.0000,1.000,2.0000,5.00
subtotal_usd,25000.0,116.187322,136.994998,3.36,38.4675,72.390,140.1600,2636.45
discount_pct,25000.0,5.630000,9.740785,0.00,0.0000,0.000,10.0000,50.00
discount_amount_usd,25000.0,6.345500,17.530764,0.00,0.0000,0.000,5.2500,421.58
shipping_fee_usd,25000.0,3.867579,3.269618,0.00,0.0000,3.990,6.9900,9.99
tax_pct,25000.0,10.677080,6.504922,0.00,8.0000,10.000,18.0000,20.00
tax_amount_usd,25000.0,11.746785,17.723056,0.00,2.3000,6.220,13.9900,303.99
total_amount_usd,25000.0,125.456186,145.635016,3.00,43.4500,78.775,149.8725,2730.88
delivery_days,25000.0,4.179480,2.548507,1.00,3.0000,4.000,5.0000,14.00


In [20]:
orders_clean["calculated_subtotal"] = (
    orders_clean["unit_price_usd"] * orders_clean["quantity"]
)

In [21]:
(
    orders_clean["subtotal_usd"].round(2)
    != orders_clean["calculated_subtotal"].round(2)
).sum()

np.int64(0)

In [22]:
orders_clean.drop(columns=["calculated_subtotal"], inplace=True)

In [23]:
calculated_discount = (
    orders_clean["subtotal_usd"]
    * orders_clean["discount_pct"]
    / 100
)

(
    orders_clean["discount_amount_usd"].round(2)
    != calculated_discount.round(2)
).sum()

np.int64(0)

In [24]:
calculated_tax = (
    (orders_clean["subtotal_usd"] - orders_clean["discount_amount_usd"])
    * orders_clean["tax_pct"]
    / 100
)

(
    orders_clean["tax_amount_usd"].round(2)
    != calculated_tax.round(2)
).sum()

np.int64(0)

In [25]:
calculated_total = (
    orders_clean["subtotal_usd"]
    - orders_clean["discount_amount_usd"]
    + orders_clean["shipping_fee_usd"]
    + orders_clean["tax_amount_usd"]
)

(
    orders_clean["total_amount_usd"].round(2)
    != calculated_total.round(2)
).sum()

np.int64(0)

In [26]:
(orders_clean["delivery_date"] < orders_clean["order_date"]).sum()

np.int64(0)

In [27]:
calculated_delivery_days = (
    orders_clean["delivery_date"] - orders_clean["order_date"]
).dt.days

In [ ]:
(
    orders_clean["delivery_days"]
    != calculated_delivery_days
).sum()

np.int64(0)

In [29]:
(
    orders_clean["year"]
    != orders_clean["order_date"].dt.year
).sum()

np.int64(0)

In [30]:
calculated_delivery_days = (
    orders_clean["delivery_date"] - orders_clean["order_date"]
).dt.days

In [31]:
calculated_quarter = (
    "Q" + orders_clean["order_date"].dt.quarter.astype(str)
)

In [32]:
(
    orders_clean["quarter"]
    != calculated_quarter
).sum()

np.int64(0)

In [33]:
calculated_day = orders_clean["order_date"].dt.day_name()

In [ ]:
(
    orders_clean["day_of_week"]
    != calculated_day
).sum()

np.int64(0)

In [36]:
categorical_cols = [
    "quarter",
    "day_of_week",
    "category",
    "payment_method",
    "device_used",
    "order_status",
    "returned",
    "is_repeat_customer"
]

for col in categorical_cols:
    print(orders_clean[col].value_counts(dropna=False))

quarter
Q1    6963
Q4    6157
Q2    5942
Q3    5938
Name: count, dtype: int64
day_of_week
Monday       3658
Sunday       3628
Saturday     3626
Tuesday      3563
Thursday     3560
Friday       3485
Wednesday    3480
Name: count, dtype: int64
category
Electronics               4526
Clothing & Apparel        3981
Home & Kitchen            3068
Books                     1962
Sports & Outdoors         1761
Beauty & Personal Care    1710
Toys & Games              1518
Food & Grocery            1472
Health & Wellness         1219
Jewelry & Accessories      973
Office Supplies            770
Automotive                 768
Pet Supplies               751
Travel & Luggage           521
Name: count, dtype: int64
payment_method
Credit Card             9522
Debit Card              5495
PayPal                  4528
UPI / Digital Wallet    2538
Buy Now Pay Later       1504
Bank Transfer            925
Cryptocurrency           488
Name: count, dtype: int64
device_used
Mobile     13989
Desktop     8090

In [37]:
print("Invalid quantity:", (orders_clean["quantity"] <= 0).sum())

print("Invalid unit price:", (orders_clean["unit_price_usd"] <= 0).sum())

print(
    "Invalid discount:",
    ((orders_clean["discount_pct"] < 0) |
     (orders_clean["discount_pct"] > 100)).sum()
)

print(
    "Invalid tax:",
    ((orders_clean["tax_pct"] < 0) |
     (orders_clean["tax_pct"] > 100)).sum()
)

print(
    "Invalid delivery days:",
    (orders_clean["delivery_days"] < 0).sum()
)

print(
    "Invalid rating:",
    (
        orders_clean["customer_rating"].notna()
        &
        ~orders_clean["customer_rating"].between(1, 5)
    ).sum()
)

Invalid quantity: 0
Invalid unit price: 0
Invalid discount: 0
Invalid tax: 0
Invalid delivery days: 0
Invalid rating: 0


In [38]:
orders_clean["returned"].unique()

array([0, 1])

In [39]:
orders_clean["is_repeat_customer"].unique()

array([1, 0])

In [40]:
orders_clean["returned"] = orders_clean["returned"].astype(bool)
orders_clean["is_repeat_customer"] = orders_clean["is_repeat_customer"].astype(bool)

In [41]:
orders_clean[["returned", "is_repeat_customer"]].dtypes

returned              bool
is_repeat_customer    bool
dtype: object

CUSTOMERS TABLE

In [7]:
customers_clean = customers.copy()

In [9]:
customers_clean.columns.tolist()

['customer_id',
 'country',
 'age',
 'gender',
 'membership_tier',
 'registration_date',
 'total_orders',
 'total_spend_usd',
 'avg_order_value_usd',
 'days_since_last_purchase',
 'preferred_category',
 'preferred_device',
 'preferred_payment_method',
 'acquisition_channel',
 'reviews_given',
 'avg_review_score',
 'returns_made',
 'wishlist_items',
 'newsletter_subscribed',
 'churned']

In [17]:
customers_clean.dtypes

customer_id                            str
country                                str
age                                  int64
gender                                 str
membership_tier                        str
registration_date           datetime64[us]
total_orders                         int64
total_spend_usd                    float64
avg_order_value_usd                float64
days_since_last_purchase             int64
preferred_category                     str
preferred_device                       str
preferred_payment_method               str
acquisition_channel                    str
reviews_given                        int64
avg_review_score                   float64
returns_made                         int64
wishlist_items                       int64
newsletter_subscribed                int64
churned                              int64
dtype: object

In [16]:
customers_clean["registration_date"] = pd.to_datetime(customers_clean["registration_date"])

In [18]:
customers_clean.head()

,customer_id,country,age,gender,membership_tier,registration_date,total_orders,total_spend_usd,avg_order_value_usd,days_since_last_purchase,preferred_category,preferred_device,preferred_payment_method,acquisition_channel,reviews_given,avg_review_score,returns_made,wishlist_items,newsletter_subscribed,churned
0,C00001,United States,40,Male,Free,2019-01-17,4,286.63,63.78,49,Food & Grocery,Mobile,Debit Card,Social Media,1,4.5,0,12,0,0
1,C00002,United States,20,Female,Free,2026-03-04,11,1245.18,107.32,126,Toys & Games,Mobile,Debit Card,Organic Search,2,2.6,1,1,0,0
2,C00003,United States,43,Female,Gold,2026-02-08,4,195.37,42.74,0,Home & Kitchen,Mobile,PayPal,Referral,0,4.8,0,0,1,0
3,C00004,United States,41,Male,Free,2025-03-19,6,99.45,15.61,6,Electronics,Desktop,PayPal,Organic Search,2,4.2,0,8,1,0
4,C00005,France,37,Other,Platinum,2024-09-10,36,2593.21,79.09,161,Clothing & Apparel,Tablet,Debit Card,Social Media,9,4.0,4,5,1,0


In [22]:
print("Rows:", len(customers_clean))
print("Unique customer IDs:", customers_clean["customer_id"].nunique())
print("Duplicate customer IDs:", customers_clean["customer_id"].duplicated().sum())
print("Missing customer IDs:", customers_clean["customer_id"].isna().sum())

Rows: 8000
Unique customer IDs: 8000
Duplicate customer IDs: 0
Missing customer IDs: 0


In [23]:
customers_clean.isna().sum()

customer_id                 0
country                     0
age                         0
gender                      0
membership_tier             0
registration_date           0
total_orders                0
total_spend_usd             0
avg_order_value_usd         0
days_since_last_purchase    0
preferred_category          0
preferred_device            0
preferred_payment_method    0
acquisition_channel         0
reviews_given               0
avg_review_score            0
returns_made                0
wishlist_items              0
newsletter_subscribed       0
churned                     0
dtype: int64

In [26]:
customer_categorical_cols = customers_clean.select_dtypes(include="object").columns

for col in customer_categorical_cols:
    print(f"\n{col.upper()}")
    print(customers_clean[col].value_counts(dropna=False))


CUSTOMER_ID
customer_id
C00001    1
C00002    1
C00003    1
C00004    1
C00005    1
         ..
C07996    1
C07997    1
C07998    1
C07999    1
C08000    1
Name: count, Length: 8000, dtype: int64

COUNTRY
country
United States     2509
United Kingdom     800
India              711
Germany            637
France             483
Canada             409
Brazil             370
Australia          323
Mexico             288
Japan              238
South Korea        170
Italy              167
Netherlands        153
Spain              152
Singapore          125
UAE                106
Poland             100
South Africa        94
Sweden              89
Turkey              76
Name: count, dtype: int64

GENDER
gender
Female    3895
Male      3866
Other      239
Name: count, dtype: int64

MEMBERSHIP_TIER
membership_tier
Free        4443
Silver      1736
Gold        1177
Platinum     644
Name: count, dtype: int64

PREFERRED_CATEGORY
preferred_category
Electronics               1505
Clothing & Appare

C:\Users\annav\AppData\Local\Temp\ipykernel_2896\3924253032.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  customer_categorical_cols = customers_clean.select_dtypes(include="object").columns


In [27]:
customers_clean.describe().T

,count,mean,min,25%,50%,75%,max,std
age,8000.0,35.616375,18.0,27.0,35.0,43.0,75.0,11.170455
registration_date,8000,2024-08-17 20:54:14.400000,2011-08-03 00:00:00,2023-12-25 00:00:00,2025-02-12 00:00:00,2025-10-13 00:00:00,2026-04-01 00:00:00,NaN
total_orders,8000.0,16.54525,1.0,5.0,12.0,23.0,79.0,14.681064
total_spend_usd,8000.0,1558.64235,4.89,336.055,845.7,1892.165,61282.48,2284.094953
avg_order_value_usd,8000.0,94.845566,5.0,44.69,72.27,118.56,1051.73,78.992885
days_since_last_purchase,8000.0,59.583875,0.0,16.0,41.0,84.0,582.0,60.610355
reviews_given,8000.0,3.22875,0.0,0.0,2.0,5.0,28.0,3.942698
avg_review_score,8000.0,4.109112,1.8,3.8,4.2,4.5,5.0,0.523992
returns_made,8000.0,0.8495,0.0,0.0,0.0,1.0,11.0,1.407337
wishlist_items,8000.0,4.457125,0.0,1.0,3.0,6.0,41.0,4.854391


In [42]:
print("Invalid age:",
      (~customers_clean["age"].between(18, 100)).sum())
print("Invalid total orders:",
      (customers_clean["total_orders"]<=0).sum())
print("Invalid total spend:",
      (customers_clean["total_spend_usd"]<=0).sum())
print("Invalid days since purchase:",
      (customers_clean["days_since_last_purchase"]<0).sum())
print("Returns great than orders:",
      (customers_clean["returns_made"]> customers_clean["total_orders"]).sum())
print("Invalid review score:",
      (~customers_clean["avg_review_score"].between(1, 5)).sum())

Invalid age: 0
Invalid total orders: 0
Invalid total spend: 0
Invalid days since purchase: 0
Returns great than orders: 0
Invalid review score: 0


In [46]:
customers_clean["newsletter_subscribed"].unique()
customers_clean["churned"].unique()

array([0, 1])

In [49]:
customers_clean["newsletter_subscribed"] = customers_clean["newsletter_subscribed"].astype(bool)
customers_clean["churned"] = customers_clean["churned"].astype(bool)

customers_clean.dtypes

customer_id                            str
country                                str
age                                  int64
gender                                 str
membership_tier                        str
registration_date           datetime64[us]
total_orders                         int64
total_spend_usd                    float64
avg_order_value_usd                float64
days_since_last_purchase             int64
preferred_category                     str
preferred_device                       str
preferred_payment_method               str
acquisition_channel                    str
reviews_given                        int64
avg_review_score                   float64
returns_made                         int64
wishlist_items                       int64
newsletter_subscribed                 bool
churned                               bool
dtype: object

In [50]:
orders_clean["order_id"].duplicated().sum()

np.int64(0)

In [52]:
customer_metrics = (
    orders_clean
    .groupby("customer_id")
    .agg(
        calculated_total_orders= ("order_id", "nunique"),
        calculated_total_spend= ("total_amount_usd", "sum"),
        calculated_returns= ("returned", "sum")
    )
    .reset_index()
)

In [53]:
customer_validation = customers_clean.merge(
    customer_metrics,
    on="customer_id",
    how= "left"
)

In [59]:
print(
   "Total orders mismatches:",
   (
       customer_validation["total_orders"] != customer_validation["calculated_total_orders"]
   ).sum()
)
print(
    "Total spend mismatches:",
    (
        customer_validation["total_spend_usd"] != customer_validation["calculated_total_spend"]
    ).sum()
)
print(
    "Returns mismatches:",
    (
        customer_validation["returns_made"] != customer_validation["calculated_returns"]
    ).sum()
)

Total orders mismatches: 7604
Total spend mismatches: 8000
Returns mismatches: 4210


In [60]:
print("Orders available in orders.csv:", orders_clean["order_id"].nunique())

print(
    "Total orders reported in customers.csv:",
    customers_clean["total_orders"].sum()
)

Orders available in orders.csv: 25000
Total orders reported in customers.csv: 132362


Validation note: Customer-level aggregate metrics in customers.csv do not fully reconcile with the available orders.csv transactions. customers.csv reports 132,362 historical orders, while orders.csv contains 25,000 transactions. Therefore, customer aggregate fields were retained and treated as broader historical metrics rather than overwritten.

In [62]:
customers_clean.to_csv(processed_path / "customers_clean.csv", index= False)
orders_clean.to_csv(processed_path / "orders_clean.csv", index=False)

In [64]:
list(processed_path.iterdir())

[WindowsPath('../data/processed/customers_clean.csv'),
 WindowsPath('../data/processed/orders_clean.csv')]

In [66]:
customers_processed = pd.read_csv(processed_path / "customers_clean.csv")
orders_processed = pd.read_csv(processed_path / "orders_clean.csv")

In [68]:
print(customers_processed.shape)
print(orders_processed.shape)

(8000, 20)
(25000, 28)
